<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Application-Deployment/blob/main/scripts/customer_data_faker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker

# 1. Initialize Faker for English data
fake = Faker("en_US")
Faker.seed(42)
np.random.seed(42)

# Generate 14,900 unique rows first (adding 100 duplicates brings total to 15,000)
NUM_UNIQUE = 14900

data = []
gender_list = ["Male", "Female"]
tier_list = ["Bronze", "Silver", "Gold", "VIP"]
tier_weights = [0.5, 0.3, 0.15, 0.05]
payment_list = ["Credit Card", "Bank Transfer", "PayPal", "Apple Pay"]
city_list = [
    "New York",
    "Los Angeles",
    "Chicago",
    "Houston",
    "Phoenix",
    "Philadelphia",
    "San Antonio",
    "San Diego",
]

for i in range(1, NUM_UNIQUE + 1):
    gender = np.random.choice(gender_list)
    name = fake.name_male() if gender == "Male" else fake.name_female()
    age = np.random.randint(19, 70)
    email = fake.email()
    signup_date = fake.date_between(start_date="-3y", end_date="today")

    annual_income = max(20000, int(np.random.normal(loc=65000, scale=15000)))
    purchase_count = np.random.poisson(lam=12)
    total_spent = round(
        purchase_count * np.random.uniform(20.0, 200.0), 2
    )
    churn_score = round(np.random.uniform(0.0, 1.0), 2)
    satisfaction_score = np.random.randint(1, 6)

    membership_tier = np.random.choice(tier_list, p=tier_weights)
    preferred_payment = np.random.choice(payment_list)
    residence_city = np.random.choice(city_list)
    is_subscribed = np.random.choice([True, False], p=[0.3, 0.7])
    app_installed = np.random.choice([True, False], p=[0.8, 0.2])

    data.append(
        {
            "Customer_ID": f"CUST_{i:05d}",
            "Name": name,
            "Gender": gender,
            "Age": age,
            "Email": email,
            "City": residence_city,
            "Signup_Date": signup_date,
            "Membership_Tier": membership_tier,
            "Annual_Income": annual_income,
            "Purchase_Count": purchase_count,
            "Total_Spent": total_spent,
            "Payment_Method": preferred_payment,
            "Satisfaction_Score": satisfaction_score,
            "Churn_Risk_Score": churn_score,
            "App_Installed": app_installed,
            "Newsletter_Subscribed": is_subscribed,
        }
    )

df = pd.DataFrame(data)

# 2. Inject 100 Duplicate Rows
duplicate_rows = df.sample(n=100, random_state=42)
df = pd.concat([df, duplicate_rows], ignore_index=True)

# 3. Inject 20 Outliers
outlier_indices = np.random.choice(df.index, size=20, replace=False)

# (1) 7 Income Outliers ($500,000 ~ $1,000,000)
df.loc[outlier_indices[:7], "Annual_Income"] = np.random.randint(
    500000, 1000000, size=7
)

# (2) 7 Purchase Count / Total Spent Outliers
df.loc[outlier_indices[7:14], "Purchase_Count"] = np.random.randint(
    300, 500, size=7
)
df.loc[outlier_indices[7:14], "Total_Spent"] = (
    df.loc[outlier_indices[7:14], "Purchase_Count"] * 500.0
)

# (3) 6 Age Outliers (Unrealistic values: 120 - 180 years old)
df.loc[outlier_indices[14:], "Age"] = np.random.randint(120, 181, size=6)

# 4. Inject Missing Values (NaN) (3% for Income, 2% for Satisfaction Score)
df.loc[np.random.rand(len(df)) < 0.03, "Annual_Income"] = np.nan
df.loc[np.random.rand(len(df)) < 0.02, "Satisfaction_Score"] = np.nan

# 5. Validation
print(f"Total rows: {len(df)}")
print(f"Duplicate rows count: {df.duplicated().sum()}")
print(f"Age outliers (>100): {(df['Age'] > 100).sum()}")
print(f"Income outliers (>$200,000): {(df['Annual_Income'] > 200000).sum()}")
print(f"Purchase Count outliers (>100): {(df['Purchase_Count'] > 100).sum()}")